# Task 179: plenopticam_lf — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Light field depth estimation using plenopticam

**Paper**: PlenoptiCam v1.0: A Light-Field Imaging Framework
**Repository**: https://github.com/hahnec/plenopticam

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 43.39 dB (sub-aperture), 8.27 dB (depth)
- **SSIM**: 0.983 (sub-aperture)
- **Evaluation**: Light field depth estimation + sub-aperture reconstruction (depth PSNR/CC, image PSNR/SSIM)

### Mathematical Formulation

**Blind Source Separation**: observed mixtures $\mathbf{x}(t) = A\mathbf{s}(t)$

**ICA** finds unmixing $W$: $\hat{\mathbf{s}} = W\mathbf{x}$ maximizing non-Gaussianity:
$$\max_{\mathbf{w}} |E\{G(\mathbf{w}^T\mathbf{x})\} - E\{G(\nu)\}|$$

**NMF**: $V \approx WH$, with $W, H \geq 0$, minimizing $D(V\|WH)$.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 179: plenopticam_lf — Light field image reconstruction
Inverse problem: raw plenoptic camera data → sub-aperture images + depth estimation

Forward model:
  A scene with known depth → render through microlens array (MLA) → raw sensor image
  Disparity d(s,t) = baseline * (u - u_center) / Z(s,t)

Inverse:
  Raw MLA image → extract sub-aperture views → estimate depth via disparity
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`lf_to_raw_mla`, `add_noise`, `extract_subapertures`, `compute_cc`, `main`

In [ ]:
def lf_to_raw_mla(lf):
    """
    Interleave sub-aperture images into a raw MLA sensor image.
    Raw pixel (i, j) = lf[i % n_ang, j % n_ang, i // n_ang, j // n_ang]
    """
    n_ang = lf.shape[0]
    s_size = lf.shape[2]
    raw_h = n_ang * s_size
    raw = np.zeros((raw_h, raw_h), dtype=np.float64)
    for u in range(n_ang):
        for v in range(n_ang):
            raw[u::n_ang, v::n_ang] = lf[u, v]
    return raw

def add_noise(raw, std=NOISE_STD):
    """Add Gaussian noise to the raw image."""
    noisy = raw + std * np.random.randn(*raw.shape)
    return np.clip(noisy, 0, 1)

# ─────────────────────────── Step 3: Inverse solver ───────────────
def extract_subapertures(raw, n_ang=N_ANGULAR):
    """De-interleave raw MLA image → (n_ang, n_ang, H, W) sub-apertures."""
    s_size = raw.shape[0] // n_ang
    lf = np.zeros((n_ang, n_ang, s_size, s_size), dtype=np.float64)
    for u in range(n_ang):
        for v in range(n_ang):
            lf[u, v] = raw[u::n_ang, v::n_ang]
    return lf

def compute_cc(gt, est):
    return float(np.corrcoef(gt.ravel(), est.ravel())[0, 1])

# ─────────────────────────── Main pipeline ────────────────────────
def main():
    os.makedirs('results', exist_ok=True)

    # 1. Create scene + depth
    print("[1/6] Creating scene and depth map ...")
    scene, gt_depth = create_scene_and_depth()
    gt_center = scene.copy()

    # 2. Forward: render light field + raw MLA image
    print("[2/6] Forward rendering light field (5×5 views) ...")
    lf = forward_render_light_field(scene, gt_depth)
    raw_clean = lf_to_raw_mla(lf)
    raw_noisy = add_noise(raw_clean)

    # 3. Inverse: extract sub-apertures + estimate depth
    print("[3/6] Extracting sub-aperture views ...")
    lf_recon = extract_subapertures(raw_noisy)
    u_c = (N_ANGULAR - 1) // 2
    recon_center = lf_recon[u_c, u_c]

    print("[4/6] Estimating depth via block matching (this may take a moment) ...")
    est_depth, disp_map = estimate_depth_block_matching(lf_recon)

    # 4. Metrics
    print("[5/6] Computing metrics ...")
    depth_psnr = compute_psnr(gt_depth, est_depth)
    depth_cc = compute_cc(gt_depth, est_depth)
    sa_psnr = compute_psnr(gt_center, recon_center)
    sa_ssim = compute_ssim(gt_center, recon_center)

    metrics = {
        "depth_psnr_dB": round(depth_psnr, 2),
        "depth_cc": round(depth_cc, 4),
        "subaperture_psnr_dB": round(sa_psnr, 2),
        "subaperture_ssim": round(sa_ssim, 4),
        "noise_std": NOISE_STD,
        "n_angular": N_ANGULAR,
        "scene_size": SCENE_SIZE,
    }

    with open('results/metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f"\n{'='*50}")
    print(f"  Depth  PSNR : {depth_psnr:.2f} dB")
    print(f"  Depth  CC   : {depth_cc:.4f}")
    print(f"  SA     PSNR : {sa_psnr:.2f} dB")
    print(f"  SA     SSIM : {sa_ssim:.4f}")
    print(f"{'='*50}\n")

    # 5. Visualization
    print("[6/6] Generating visualization ...")
    visualize(gt_center, raw_noisy, recon_center,
              gt_depth, est_depth, 'results/reconstruction_result.png')

    # 6. Save arrays
    np.save('results/ground_truth.npy', gt_depth)
    np.save('results/reconstruction.npy', est_depth)
    print("[DONE] All results saved to results/")

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `create_scene_and_depth`, `forward_render_light_field`

In [ ]:
# ─────────────────────────── Step 1: Create scene + depth ─────────
def create_scene_and_depth(size=SCENE_SIZE):
    """Create a 2D scene with 3 depth layers and corresponding depth map."""
    scene = np.zeros((size, size), dtype=np.float64)
    depth = np.full((size, size), 5.0, dtype=np.float64)  # background depth

    # Background: smooth gradient
    yy, xx = np.mgrid[0:size, 0:size]
    scene = 0.3 + 0.3 * (xx / size) + 0.1 * np.sin(2 * np.pi * yy / size * 3)

    # Object 1: circle at depth Z=2.0 (near, large disparity)
    cy, cx, r = size // 3, size // 3, size // 6
    mask_circle = ((yy - cy) ** 2 + (xx - cx) ** 2) < r ** 2
    scene[mask_circle] = 0.85
    depth[mask_circle] = 2.0

    # Object 2: square at depth Z=3.5 (mid)
    sy, sx = int(size * 0.55), int(size * 0.55)
    half = size // 8
    mask_sq = (np.abs(yy - sy) < half) & (np.abs(xx - sx) < half)
    scene[mask_sq] = 0.55
    depth[mask_sq] = 3.5

    # Add mild texture so block matching has features to lock onto
    texture = 0.05 * np.random.RandomState(123).randn(size, size)
    texture = gaussian_filter(texture, sigma=1.0)
    scene = np.clip(scene + texture, 0, 1)

    return scene, depth

# ─────────────────────────── Step 2: Forward operator ─────────────
def forward_render_light_field(scene, depth, n_ang=N_ANGULAR, baseline=BASELINE):
    """
    Render a 4-D light field L[u, v, s, t] from a 2-D scene + depth map.

    For sub-aperture (u, v), the scene is shifted by
        dx = baseline * (u - u_c) / Z(s,t)
        dy = baseline * (v - v_c) / Z(s,t)
    """
    size = scene.shape[0]
    u_c = (n_ang - 1) / 2.0
    v_c = u_c
    lf = np.zeros((n_ang, n_ang, size, size), dtype=np.float64)

    for u in range(n_ang):
        for v in range(n_ang):
            du = baseline * (u - u_c)
            dv = baseline * (v - v_c)
            # Per-pixel disparity from depth
            disp_x = du / depth   # shape (size, size)
            disp_y = dv / depth

            # Approximate: use mean disparity per depth layer for efficiency
            shifted = np.zeros_like(scene)
            for z_val in np.unique(depth):
                mask = depth == z_val
                dx = du / z_val
                dy = dv / z_val
                layer = np.where(mask, scene, 0.0)
                shifted_layer = ndi_shift(layer, [dy, dx], order=1, mode='nearest')
                shifted += shifted_layer

            lf[u, v] = shifted

    return lf

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `estimate_depth_block_matching`

In [ ]:
def estimate_depth_block_matching(lf, baseline=BASELINE,
                                  patch_half=PATCH_HALF,
                                  disp_range=DISP_RANGE):
    """
    Estimate depth by searching over candidate depth values.

    Forward model: sub-aperture (u,v) sees the scene shifted by
        dx = baseline*(u - u_c) / Z,   dy = baseline*(v - v_c) / Z

    So if we warp view (u,v) back by that shift, it should align with center.
    We discretise Z into candidates and pick the Z that gives best NCC.
    """
    n_ang = lf.shape[0]
    size = lf.shape[2]
    u_c = (n_ang - 1) / 2.0
    v_c = u_c
    center = lf[int(u_c), int(v_c)]

    # Candidate depth values — densely sample the range we know is relevant
    z_candidates = np.linspace(1.5, 6.5, 60)
    n_z = len(z_candidates)

    win = 2 * patch_half + 1
    c_mean = uniform_filter(center, size=win, mode='nearest')
    c_sq_mean = uniform_filter(center ** 2, size=win, mode='nearest')
    c_std = np.sqrt(np.maximum(c_sq_mean - c_mean ** 2, 1e-12))

    cost_vol = np.zeros((n_z, size, size), dtype=np.float64)
    count = 0

    for u in range(n_ang):
        for v in range(n_ang):
            du = u - u_c
            dv = v - v_c
            if du == 0 and dv == 0:
                continue
            count += 1
            ref = lf[u, v]
            for zi, z_val in enumerate(z_candidates):
                # The view (u,v) was created by shifting the scene by
                # (baseline*du/Z, baseline*dv/Z).
                # To undo, shift the view by the NEGATIVE of that.
                sx = -baseline * du / z_val   # column shift
                sy = -baseline * dv / z_val   # row shift
                warped = ndi_shift(ref, [sy, sx], order=1, mode='nearest')
                w_mean = uniform_filter(warped, size=win, mode='nearest')
                w_sq_mean = uniform_filter(warped ** 2, size=win, mode='nearest')
                w_std = np.sqrt(np.maximum(w_sq_mean - w_mean ** 2, 1e-12))
                cross = uniform_filter(center * warped, size=win, mode='nearest')
                ncc = (cross - c_mean * w_mean) / (c_std * w_std + 1e-12)
                cost_vol[zi] += ncc

    # Winner-take-all
    best_idx = np.argmax(cost_vol, axis=0)
    est_depth = z_candidates[best_idx]

    # Sub-pixel refinement via parabola
    for s in range(size):
        for t in range(size):
            idx = best_idx[s, t]
            if 0 < idx < n_z - 1:
                c0 = cost_vol[idx - 1, s, t]
                c1 = cost_vol[idx, s, t]
                c2 = cost_vol[idx + 1, s, t]
                denom = 2.0 * (c0 - 2 * c1 + c2)
                if abs(denom) > 1e-12:
                    offset = (c0 - c2) / denom
                    refined_idx = idx + np.clip(offset, -0.5, 0.5)
                    est_depth[s, t] = np.interp(refined_idx,
                                                np.arange(n_z), z_candidates)

    # Mild smoothing to denoise
    est_depth = gaussian_filter(est_depth, sigma=1.2)

    return est_depth, best_idx

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `compute_ssim`, `visualize`

In [ ]:
# ─────────────────────────── Step 4: Metrics ──────────────────────
def compute_psnr(gt, est):
    mse = np.mean((gt - est) ** 2)
    if mse < 1e-15:
        return 100.0
    data_range = np.max(gt) - np.min(gt)
    if data_range < 1e-15:
        data_range = 1.0
    return 10.0 * np.log10(data_range ** 2 / mse)

def compute_ssim(gt, est):
    data_range = max(gt.max() - gt.min(), est.max() - est.min(), 1e-6)
    return float(ssim(gt, est, data_range=data_range))

# ─────────────────────────── Step 5: Visualization ────────────────
def visualize(gt_scene, raw_mla, recon_center, gt_depth, est_depth, save_path):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # (a) GT center sub-aperture
    ax = axes[0, 0]
    ax.imshow(gt_scene, cmap='gray', vmin=0, vmax=1)
    ax.set_title('(a) GT Center View')
    ax.axis('off')

    # (b) Raw MLA image (small patch)
    ax = axes[0, 1]
    patch_size = min(60 * N_ANGULAR, raw_mla.shape[0])
    ax.imshow(raw_mla[:patch_size, :patch_size], cmap='gray', vmin=0, vmax=1)
    ax.set_title('(b) Raw MLA Image (patch)')
    ax.axis('off')

    # (c) Reconstructed center sub-aperture
    ax = axes[0, 2]
    ax.imshow(recon_center, cmap='gray', vmin=0, vmax=1)
    ax.set_title('(c) Reconstructed Center View')
    ax.axis('off')

    # (d) GT depth map
    ax = axes[1, 0]
    vmin_d, vmax_d = gt_depth.min(), gt_depth.max()
    im_d = ax.imshow(gt_depth, cmap='viridis', vmin=vmin_d, vmax=vmax_d)
    ax.set_title('(d) GT Depth Map')
    ax.axis('off')
    plt.colorbar(im_d, ax=ax, fraction=0.046)

    # (e) Estimated depth map
    ax = axes[1, 1]
    im_e = ax.imshow(est_depth, cmap='viridis', vmin=vmin_d, vmax=vmax_d)
    ax.set_title('(e) Estimated Depth Map')
    ax.axis('off')
    plt.colorbar(im_e, ax=ax, fraction=0.046)

    # (f) Depth error map
    ax = axes[1, 2]
    err = np.abs(gt_depth - est_depth)
    im_f = ax.imshow(err, cmap='hot')
    ax.set_title('(f) Depth Error Map')
    ax.axis('off')
    plt.colorbar(im_f, ax=ax, fraction=0.046)

    plt.suptitle('Task 179: Light Field Reconstruction (plenopticam_lf)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Visualization saved to {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
os.makedirs('results', exist_ok=True)

### 1. Create scene + depth

Intermediate processing step in the pipeline.

In [ ]:
# 1. Create scene + depth
print("[1/6] Creating scene and depth map ...")
scene, gt_depth = create_scene_and_depth()
gt_center = scene.copy()

### 2. Forward: render light field + raw MLA image

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Forward: render light field + raw MLA image
print("[2/6] Forward rendering light field (5×5 views) ...")
lf = forward_render_light_field(scene, gt_depth)
raw_clean = lf_to_raw_mla(lf)
raw_noisy = add_noise(raw_clean)

### 3. Inverse: extract sub-apertures + estimate depth

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Inverse: extract sub-apertures + estimate depth
print("[3/6] Extracting sub-aperture views ...")
lf_recon = extract_subapertures(raw_noisy)
u_c = (N_ANGULAR - 1) // 2
recon_center = lf_recon[u_c, u_c]

print("[4/6] Estimating depth via block matching (this may take a moment) ...")
est_depth, disp_map = estimate_depth_block_matching(lf_recon)

### 4. Metrics

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 4. Metrics
print("[5/6] Computing metrics ...")
depth_psnr = compute_psnr(gt_depth, est_depth)
depth_cc = compute_cc(gt_depth, est_depth)
sa_psnr = compute_psnr(gt_center, recon_center)
sa_ssim = compute_ssim(gt_center, recon_center)

metrics = {
    "depth_psnr_dB": round(depth_psnr, 2),
    "depth_cc": round(depth_cc, 4),
    "subaperture_psnr_dB": round(sa_psnr, 2),
    "subaperture_ssim": round(sa_ssim, 4),
    "noise_std": NOISE_STD,
    "n_angular": N_ANGULAR,
    "scene_size": SCENE_SIZE,
}

with open('results/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n{'='*50}")
print(f"  Depth  PSNR : {depth_psnr:.2f} dB")
print(f"  Depth  CC   : {depth_cc:.4f}")
print(f"  SA     PSNR : {sa_psnr:.2f} dB")
print(f"  SA     SSIM : {sa_ssim:.4f}")
print(f"{'='*50}\n")

### 5. Visualization

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 5. Visualization
print("[6/6] Generating visualization ...")
visualize(gt_center, raw_noisy, recon_center,
          gt_depth, est_depth, 'results/reconstruction_result.png')

### 6. Save arrays

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 6. Save arrays
np.save('results/ground_truth.npy', gt_depth)
np.save('results/reconstruction.npy', est_depth)
print("[DONE] All results saved to results/")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **plenopticam_lf**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=43.39 dB (sub-aperture), 8.27 dB (depth), SSIM=0.983 (sub-aperture))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: PlenoptiCam v1.0: A Light-Field Imaging Framework
- Repository: https://github.com/hahnec/plenopticam
